# 1. Content Understanding: extract, chunk, and embed claim evidence
Non-spark environment: use this repository's `.venv` and root `.env`. Use native documentSearch for PDFs, document scans, and data-rich diagrams; use native imageSearch for photos. Keep native Markdown, visual descriptions, and chart/diagram content.

Review each step before running it. CU extraction and embeddings are paid calls. Visual observations are not findings of cause, timing, liability, or coverage. This notebook does not create or load a Search index.

In [ ]:
import os, json, hashlib, time
from pathlib import Path, PurePosixPath, PureWindowsPath
from urllib.parse import urlsplit, unquote
from tempfile import TemporaryDirectory
from dotenv import load_dotenv, find_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

env_file = find_dotenv(usecwd=True)
load_dotenv(env_file, override=True)
ROOT = Path(env_file).parent if env_file else Path.cwd()

credential = DefaultAzureCredential()

azure_openai_embedding_deployment = os.getenv('AZURE_OPENAI_EMBEDDING_DEPLOYMENT', 'text-embedding-3-large')
azure_openai_model_dimensions = int(os.getenv('AZURE_OPENAI_EMBEDDING_DIMENSIONS', '3072'))
claim_location = os.getenv('CLAIM_FOLDER', 'data/claims/claim-001')
claim_id = PurePosixPath(claim_location.replace('\\', '/').rstrip('/')).name
output_location = os.getenv('CU_OUTPUT_ROOT', 'output/claims/cu').rstrip('/\\') + '/' + claim_id
print('Claim:', claim_id, '| Content Understanding')

## Compact display settings
Display limits never truncate the working DataFrame, saved records, full content, or vectors.

In [ ]:
import pandas as pd
from IPython.display import Markdown, display
pd.set_option('display.max_rows', 5)
pd.set_option('display.max_columns', 15)
pd.set_option('display.max_colwidth', 160)
pd.set_option('display.max_seq_items', 3)

def ShowPreview(frame):
    preview = frame.head(5).copy()
    for column in preview.columns:
        if column.endswith('_vector'):
            preview[column] = preview[column].map(lambda vector: f'{len(vector)} dimensions')
        else:
            preview[column] = preview[column].map(lambda value: str(value)[:160])
    display(preview)

## Step 1: Clear generated outputs and inspect source files
Start each run with an empty generated output folder for this claim, then read the original inputs. Use a local folder, a Unity Catalog `/Volumes/...` path, or an `abfss://...` ADLS folder. Input files are never deleted. Output cleanup is restricted to a dedicated folder separate from inputs, the repository, and the home directory. Direct ADLS stages only input files; all extraction, visual analysis, and embeddings run again.

In [ ]:
from azure.storage.filedatalake import FileSystemClient
from shutil import rmtree
staging_folders, storage_clients = [], []

def StageFolder(location, required):
    if location.startswith('abfss://'):
        parsed = urlsplit(location)
        container, host = parsed.netloc.split('@')
        prefix = unquote(parsed.path).strip('/')
        client = FileSystemClient(account_url=f'https://{host}', file_system_name=container, credential=credential)
        storage_clients.append(client)
        client.get_file_system_properties()
        staging = TemporaryDirectory(prefix='azure-rag-workshop-')
        staging_folders.append(staging)
        folder = Path(staging.name)
        if required:
            for item in client.get_paths(path=prefix, recursive=True):
                if item.is_directory:
                    continue
                relative = PurePosixPath(item.name).relative_to(PurePosixPath(prefix))
                target = folder.joinpath(*relative.parts).resolve()
                target.relative_to(folder)
                target.parent.mkdir(parents=True, exist_ok=True)
                with target.open('wb') as stream:
                    client.get_file_client(item.name).download_file().readinto(stream)
        return folder, (client, prefix), f'abfss://{container}@{host}/{prefix}'
    folder = (ROOT / location).expanduser().resolve(strict=required)
    return folder, None, str(folder)

def SourceReference(root, relative):
    return root.rstrip('/') + '/' + relative if root.startswith('abfss://') else str(Path(root) / relative)

claim_folder, input_remote, claim_source = StageFolder(claim_location, required=True)
output_folder, output_remote, output_source = StageFolder(output_location, required=False)
input_ref, output_ref = [value.replace('\\', '/').rstrip('/') for value in (claim_source, output_source)]
if os.name == 'nt':
    input_ref, output_ref = input_ref.casefold(), output_ref.casefold()
if (input_ref == output_ref or input_ref.startswith(output_ref + '/') or output_ref.startswith(input_ref + '/')
        or (output_remote and not output_remote[1])
        or (not output_remote and any(output_folder == folder or output_folder in folder.parents for folder in (ROOT, Path.home())))):
    raise ValueError('Use a dedicated output folder separate from inputs, the repository, and the home directory.')
if output_remote:
    client, prefix = output_remote
    directory = client.get_directory_client(prefix)
    if directory.exists():
        directory.delete_directory()
elif output_folder.exists():
    rmtree(output_folder)
output_folder.mkdir(parents=True, exist_ok=True)
print('Cleared generated outputs:', output_source)
(output_folder / 'attachments').mkdir(exist_ok=True)
persisted_directories = set()

def PersistFile(path):
    if output_remote:
        client, prefix = output_remote
        remote_name = prefix + '/' + path.relative_to(output_folder).as_posix()
        parts = PurePosixPath(remote_name).parent.parts
        for count in range(1, len(parts) + 1):
            directory = '/'.join(parts[:count])
            if directory not in persisted_directories:
                directory_client = client.get_directory_client(directory)
                if not directory_client.exists():
                    directory_client.create_directory()
                persisted_directories.add(directory)
        with path.open('rb') as stream:
            client.get_file_client(remote_name).upload_data(stream, overwrite=True)

files = sorted(path for path in claim_folder.rglob('*') if path.is_file())
file_df = pd.DataFrame([{'filename': path.name, 'extension': path.suffix.lower(), 'bytes': path.stat().st_size} for path in files])
ShowPreview(file_df)
print('Claim:', claim_id, '| Source files:', len(files), '| Output:', output_source)

## Step 2: Read MSG bodies and attachments locally; deduplicate bytes
MSG files are parsed locally, including nested messages. Every occurrence's attachments are visited, even when the email body repeats. Identical document bytes share one CU analysis while retaining source and parent-email links. Unsupported loose files are reported.

In [ ]:
import extract_msg
from extract_msg.msg_classes import MSGFile
documents, emails, skipped = {}, {}, []
supported_documents = {'.pdf', '.jpg', '.jpeg', '.png'}

def AddDocument(data, filename, source, parent_email=None):
    extension = Path(filename).suffix.lower()
    if extension not in supported_documents:
        raise ValueError(f'Unsupported attachment: {source}. Convert it deliberately before continuing.')
    document_hash = hashlib.sha256(data).hexdigest()
    relative = 'attachments/' + document_hash + extension if parent_email else '/'.join(PurePosixPath(source).parts[1:])
    evidence_path = output_folder / relative if parent_email else claim_folder / relative
    source_path = SourceReference(output_source if parent_email else claim_source, relative)
    item = documents.setdefault(document_hash, {'data': data, 'filename': filename, 'local_path': evidence_path, 'source_path': source_path, 'source_type': 'document' if extension == '.pdf' else 'image', 'source_files': [], 'parent_emails': []})
    item['source_files'].append(source)
    if parent_email and parent_email not in item['parent_emails']:
        item['parent_emails'].append(parent_email)
    if parent_email:
        evidence_path.write_bytes(data)
        PersistFile(evidence_path)

def ReadMessage(message, source, parent_email=None):
    if message.body is None and message.htmlBody:
        raise ValueError(f'MSG has HTML but no parsed text body: {source}. Inspect it before dropping its body.')
    headers = {'subject': message.subject or '', 'sender': message.sender or '', 'to': message.to or '', 'date': str(message.date)}
    body = (message.body or '').replace('\r\n', '\n')
    email_id = hashlib.sha256(json.dumps({**headers, 'body': body}, sort_keys=True).encode()).hexdigest()
    relative = '/'.join(PurePosixPath(source.split('::', 1)[0]).parts[1:])
    item = emails.setdefault(email_id, {'filename': PureWindowsPath(source.rsplit('::', 1)[-1]).name, 'local_path': claim_folder / relative, 'source_path': SourceReference(claim_source, relative), 'content': '# ' + headers['subject'] + '\n\n' + '\n'.join(f'{key}: {value}' for key, value in headers.items()) + '\n\n' + body, 'source_files': [], 'parent_emails': []})
    item['source_files'].append(source)
    if parent_email and parent_email not in item['parent_emails']:
        item['parent_emails'].append(parent_email)
    for number, attachment in enumerate(message.attachments):
        name = PureWindowsPath(str(attachment.getFilename() or f'attachment-{number}')).name
        data = attachment.data
        if isinstance(data, MSGFile):
            ReadMessage(data, f'{source}::{name}', email_id)
        elif isinstance(data, bytes):
            AddDocument(data, name, f'{source}::{name}', email_id)
        else:
            raise TypeError(f'Unsupported attachment data: {source}::{name}')

In [ ]:
for path in files:
    source = f'{claim_id}/{path.relative_to(claim_folder).as_posix()}'
    if path.suffix.lower() in supported_documents:
        AddDocument(path.read_bytes(), path.name, source)
    elif path.suffix.lower() != '.msg':
        skipped.append({'source': source, 'reason': 'outside PDF/JPG/PNG/MSG walkthrough'})
for path in files:
    if path.suffix.lower() == '.msg':
        with extract_msg.openMsg(str(path)) as message:
            ReadMessage(message, f'{claim_id}/{path.relative_to(claim_folder).as_posix()}')
document_df = pd.DataFrame([{'filename': item['filename'], 'source_files': item['source_files'], 'parent_emails': item['parent_emails'], 'occurrences': len(item['source_files'])} for item in documents.values()])
ShowPreview(document_df)
ShowPreview(pd.DataFrame([{'filename': item['filename'], 'content': item['content']} for item in emails.values()]))
if skipped:
    ShowPreview(pd.DataFrame(skipped))
print('Unsupported loose files:', len(skipped))
print('Unique documents:', len(documents), '| Unique emails:', len(emails))
print('Duplicate document analyses avoided:', sum(len(item['source_files']) - 1 for item in documents.values()))

## Review extraction routes before making paid calls
Classify images by purpose, not by whether OCR finds a character. PDFs and image scans of documents use documentSearch. Photos use imageSearch, even when annotated. The demo chart and access diagram use documentSearch so we can retain native figure data. Edit `document_image_names` for your own scans, charts, and diagrams. This is a visible routing choice, not an automatic classifier. Embedded PDF figures stay within documentSearch; no separate crop calls.

In [ ]:
document_image_names = {'survey-chart.png', 'site-access.png'}
document_analyzer_id = 'prebuilt-documentSearch'
image_analyzer_id = 'prebuilt-imageSearch'
analyzer_routes = {key: document_analyzer_id if item['source_type'] == 'document' or item['filename'] in document_image_names else image_analyzer_id for key, item in documents.items()}
route_df = pd.DataFrame([{'filename': item['filename'], 'source_type': item['source_type'], 'analyzer_id': analyzer_routes[key]} for key, item in documents.items()], columns=['filename', 'source_type', 'analyzer_id'])
ShowPreview(route_df)
ShowPreview(route_df.loc[route_df['source_type'].eq('image')])
print('Unique files for CU:', len(analyzer_routes), '| Routes:', route_df['analyzer_id'].value_counts().to_dict())

## Step 3: Extract native content (paid calls on every run)
Call the reviewed native CU analyzer for each unique file on every run. Save complete native responses for inspection; existing files never skip a call. No custom analyzer, prompt, or Markdown reconstruction is used.

In [ ]:
from azure.core.serialization import AzureJSONEncoder
from azure.ai.contentunderstanding import ContentUnderstandingClient
from azure.ai.contentunderstanding.models import AnalysisInput
extraction_client = None
extraction_results, extraction_summary = {}, []
if documents:
    extraction_endpoint = os.environ['APP_CONTENT_UNDERSTANDING_ENDPOINT'].rstrip('/')
    api_version = os.getenv('CU_API_VERSION', '2025-11-01')
    resource_fingerprint = hashlib.sha256(extraction_endpoint.lower().encode()).hexdigest()[:16]
    results_folder = output_folder / 'analysis-results' / resource_fingerprint
    results_folder.mkdir(parents=True, exist_ok=True)
    extraction_client = ContentUnderstandingClient(endpoint=extraction_endpoint, credential=credential, api_version=api_version)
for document_hash, item in documents.items():
    start = time.time()
    analyzer_id = analyzer_routes[document_hash]
    result_file = results_folder / f'{analyzer_id}_{api_version}_{document_hash}.json'
    poller = extraction_client.begin_analyze(analyzer_id=analyzer_id, inputs=[AnalysisInput(data=item['data'])])
    result = poller.result().as_dict()
    result_file.write_text(json.dumps(result, indent=2, cls=AzureJSONEncoder), encoding='utf-8')
    PersistFile(result_file)
    extraction_results[document_hash] = result
    extraction_summary.append({'filename': item['filename'], 'analyzer_id': analyzer_id, 'seconds': round(time.time() - start, 2)})
ShowPreview(pd.DataFrame(extraction_summary))
print('Documents analyzed this run:', len(extraction_results))

## Step 4: Review native output
Save full native document Markdown, image summaries, and figure text. Figure text includes the native description and any returned chart/diagram content; descriptions are not rewritten. Preview only one document excerpt and one figure. Visual descriptions are observations, not proof of cause, timing, liability, or coverage.

In [ ]:
def FigureText(figure):
    parts = [figure.get('description', '')]
    if figure.get('content'):
        value = figure['content']
        parts.append(value if isinstance(value, str) else json.dumps(value, ensure_ascii=False))
    return '\n\n'.join(part for part in parts if part)

for document_hash, result in extraction_results.items():
    for number, content in enumerate(result['contents']):
        native_text = (content.get('fields') or {}).get('Summary', {}).get('valueString', '') if analyzer_routes[document_hash] == image_analyzer_id else content.get('markdown') or ''
        native_file = output_folder / f'{document_hash}_{number}.md'
        native_file.write_text(native_text, encoding='utf-8')
        PersistFile(native_file)
        for figure in content.get('figures') or []:
            figure_file = output_folder / f"{document_hash}_figure_{figure['id']}.md"
            figure_file.write_text(FigureText(figure), encoding='utf-8')
            PersistFile(figure_file)
if extraction_results:
    example_hash = next((key for key, result in extraction_results.items() if any(content.get('figures') for content in result['contents'])), next(iter(extraction_results)))
    example_contents = extraction_results[example_hash]['contents']
    example = next((content for content in example_contents if content.get('figures')), example_contents[0])
    print('Example:', documents[example_hash]['filename'])
    example_text = example.get('markdown') or (example.get('fields') or {}).get('Summary', {}).get('valueString', '')
    display(Markdown(example_text[:1200]))
    ShowPreview(pd.DataFrame([{'id': figure['id'], 'native_figure_text': FigureText(figure)} for figure in (example.get('figures') or [])[:1]]))
    ShowPreview(pd.DataFrame([{'filename': documents[key]['filename'], 'image_observation': content['fields']['Summary']['valueString']} for key, result in extraction_results.items() if analyzer_routes[key] == image_analyzer_id for content in result['contents']]))
else:
    print('Email-body-only claim; no extraction API was called.')

## Step 5: Native CU content -> evidence DataFrame
Split document and email Markdown by headings. Keep imageSearch summaries as image-description rows. Keep documentSearch figures as separately citable rows containing their full native description and any chart/diagram content. This includes images embedded in PDFs. No visual content is replaced with OCR-only text.

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

def MdFormatting(markdown):
    text_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=[('#', 'Title'), ('##', 'Header 1')], strip_headers=False)
    return pd.DataFrame([{'title': chunk.metadata.get('Title', ''), 'header': chunk.metadata.get('Header 1', ''), 'content': chunk.page_content} for chunk in text_splitter.split_text(markdown)], columns=['title', 'header', 'content'])

start = time.time()
rows = []
for document_hash, result in extraction_results.items():
    item = documents[document_hash]
    evidence = {'document_hash': document_hash, 'filename': item['filename'], 'source_path': item['source_path'], 'source_type': item['source_type'], 'source_files': item['source_files'], 'parent_emails': item['parent_emails'], 'analyzer_id': analyzer_routes[document_hash], 'analyzer_api_version': result['apiVersion']}
    previous_rows = len(rows)
    for content in result['contents']:
        if evidence['analyzer_id'] == image_analyzer_id:
            observation = content['fields']['Summary']['valueString']
            rows.append({**evidence, 'title': item['filename'], 'header': 'Visual observation; not a causation or coverage finding', 'content': observation, 'content_kind': 'image_description', 'figure_id': '', 'source_region': ''})
            continue
        for chunk in MdFormatting(content.get('markdown') or '').to_dict(orient='records'):
            rows.append({**evidence, **chunk, 'content_kind': 'document_text', 'figure_id': '', 'source_region': content.get('source') or ''})
        for figure in content.get('figures') or []:
            figure_text = FigureText(figure)
            if figure_text:
                rows.append({**evidence, 'title': item['filename'], 'header': f"CU figure {figure['id']}", 'content': figure_text, 'content_kind': 'figure_description', 'figure_id': figure['id'], 'source_region': figure.get('source') or ''})
    if not any(row['content'].strip() for row in rows[previous_rows:]):
        raise ValueError(f"No searchable content in {item['filename']}; inspect its native extraction result.")
for email_id, item in emails.items():
    for chunk in MdFormatting(item['content']).to_dict(orient='records'):
        rows.append({**chunk, 'document_hash': email_id, 'filename': item['filename'], 'source_path': item['source_path'], 'source_type': 'email', 'content_kind': 'email_text', 'figure_id': '', 'source_region': '', 'source_files': item['source_files'], 'parent_emails': item['parent_emails'], 'analyzer_id': 'local-msg', 'analyzer_api_version': ''})
df = pd.DataFrame(rows)
df['title'] = df.apply(lambda row: row['title'] or row['filename'], axis=1)
ShowPreview(df[['filename', 'content_kind', 'title', 'content']])
print('Chunks:', len(df), '| Seconds:', round(time.time() - start, 2), '| Evidence kinds:', df['content_kind'].value_counts().to_dict())

## Step 6: Count tokens for the exact embedding input
Use the original title + section + evidence-kind + content input. Review the largest chunk before embedding; values are not truncated.

In [ ]:
import tiktoken
encoding = tiktoken.get_encoding('cl100k_base')
def NumberTokens(text):
    return len(encoding.encode(text, disallowed_special=()))

df['embedding_text'] = df.apply(lambda row: '\n\n'.join([f"Title: {row['title']}", f"Section: {row['header']}", f"Evidence kind: {row['content_kind']}", row['content']]), axis=1)
df['char_len'] = df['embedding_text'].apply(len)
df['token_len'] = df['embedding_text'].apply(NumberTokens)
ShowPreview(df[['filename', 'title', 'char_len', 'token_len']])
print('Chunks:', len(df), '| Largest chunk:', df['token_len'].max(), 'tokens')

## Step 7: Review inputs and create the embedding client
Use the existing embedding deployment and Entra credential. Creating the client does not generate embeddings.

In [ ]:
from openai import OpenAI
foundry_openai_endpoint = os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')
aoai_client = OpenAI(base_url=f'{foundry_openai_endpoint}/openai/v1/', api_key=get_bearer_token_provider(credential, 'https://ai.azure.com/.default'))

def GenerateVector(text):
    response = aoai_client.embeddings.create(input=text, model=azure_openai_embedding_deployment, dimensions=azure_openai_model_dimensions)
    return response.data[0].embedding

ShowPreview(df[['filename', 'content_kind', 'embedding_text', 'token_len']])
print('One embedding per row:', len(df), '| Deployment:', azure_openai_embedding_deployment, '| Dimensions:', azure_openai_model_dimensions)

## Step 8: Generate content vectors (paid calls)
One vector per evidence row; no separate title vector. Full vectors remain in memory but are never printed.

In [ ]:
start = time.time()
df['content_vector'] = df['embedding_text'].apply(GenerateVector)
print('Chunks embedded:', len(df), '| Seconds:', round(time.time() - start, 2))
ShowPreview(df[['filename', 'content_vector']])

## Step 9: Add the original deterministic identity and provenance
IDs remain the hash of claim/document/chunk. A claim filter scopes retrieval; it is not authorization.

In [ ]:
df['claim_id'] = claim_id
df['chunk_id'] = df.groupby('document_hash', sort=False).cumcount().add(1)
df['id'] = df.apply(lambda row: hashlib.sha256(f"{claim_id}:{row['document_hash']}:{row['chunk_id']}".encode()).hexdigest(), axis=1)
ShowPreview(df[['claim_id', 'filename', 'source_type', 'content_kind', 'chunk_id', 'figure_id', 'analyzer_id']])

## Step 10: Review the same 18-field index-ready DataFrame
The CU index and loader consume these same 18 fields. Native image summaries and figure descriptions remain distinguishable from document and email text.

In [ ]:
index_columns = ['id', 'claim_id', 'document_hash', 'chunk_id', 'title', 'header', 'content', 'source_type', 'content_kind', 'filename', 'source_path', 'source_files', 'parent_emails', 'figure_id', 'source_region', 'analyzer_id', 'analyzer_api_version', 'content_vector']
index_df = df[index_columns].copy()
ShowPreview(index_df)
print('Rows ready to load:', len(index_df), '| Columns:', index_df.columns.tolist())

## Step 11: Save full records
Save the full content, vectors, and provenance. Notebook 3 reads this JSON directly. No Search operations occur.

In [ ]:
processed_file = output_folder / 'processed-evidence-v1.json'
index_df.to_json(processed_file, orient='records', indent=2, force_ascii=False, double_precision=15)
PersistFile(processed_file)
print('Saved', len(index_df), 'full records to:', output_source)
print('Next: run 02_create_index.ipynb, then 03_populate_index.ipynb.')

## Cleanup
Close clients and remove only temporary ADLS staging folders. Durable originals and outputs remain. Run this cell when done, including after an earlier cell fails.

In [ ]:
for name in ['extraction_client', 'aoai_client', 'credential']:
    client = globals().get(name)
    if client is not None:
        client.close()
for client in globals().get('storage_clients', []):
    client.close()
for staging in globals().get('staging_folders', []):
    staging.cleanup()
print('Clients closed; temporary staging removed. Durable data was not deleted.')